# Part B — 03 Model Training

This notebook trains multiple classification models to predict whether
an AI patent will become high impact.

Models are trained on patents from 2021–2022 and compared using the
2023 validation set.

The 2024 test set remains untouched for final evaluation.

Models considered:

- Logistic Regression
- Decision Tree
- Random Forest

In [1]:
from pathlib import Path

import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

ROOT = Path.cwd().parent

PROCESSED_DIR = ROOT / "Data" / "processed"
TABLES_DIR = ROOT / "Outputs" / "tables"

DATA_PATH = (
    PROCESSED_DIR /
    "01_part_b_target_dataset.parquet"
)

df = pd.read_parquet(DATA_PATH)

print("Rows:", f"{len(df):,}")
print("Columns:", df.shape[1])

Rows: 49,556
Columns: 26


In [2]:
numeric_features = [
    "inventor_count",
    "assignee_count",
    "cpc_count",
    "backward_citation_count",
    "title_word_count",
    "abstract_word_count",
    "claims_text_length",
    "claims_word_count",
    "filing_to_grant_days"
]

categorical_features = [
    "cpc_group",
    "assignee_country"
]

feature_columns = (
    numeric_features +
    categorical_features
)

target = "high_impact"

X = df[feature_columns].copy()
y = df[target].copy()

In [3]:
train_mask = df["grant_year"].isin([2021, 2022])
validation_mask = df["grant_year"] == 2023
test_mask = df["grant_year"] == 2024

X_train = X.loc[train_mask].copy()
y_train = y.loc[train_mask].copy()

X_val = X.loc[validation_mask].copy()
y_val = y.loc[validation_mask].copy()

X_test = X.loc[test_mask].copy()
y_test = y.loc[test_mask].copy()

print("Training:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

Training: (26465, 11)
Validation: (17597, 11)
Test: (5494, 11)


In [4]:
numeric_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)

categorical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_transformer,
            numeric_features
        ),
        (
            "categorical",
            categorical_transformer,
            categorical_features
        )
    ]
)

In [5]:
models = {

    "Logistic Regression":
        LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            random_state=42
        ),

    "Decision Tree":
        DecisionTreeClassifier(
            class_weight="balanced",
            random_state=42
        ),

    "Random Forest":
        RandomForestClassifier(
            n_estimators=300,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        )
}

In [6]:
def train_and_evaluate(
    model_name,
    classifier
):

    pipeline = Pipeline(
        steps=[
            (
                "preprocessor",
                preprocessor
            ),
            (
                "classifier",
                classifier
            )
        ]
    )

    pipeline.fit(
        X_train,
        y_train
    )

    predictions = pipeline.predict(
        X_val
    )

    probabilities = pipeline.predict_proba(
        X_val
    )[:, 1]

    results = {
        "Model": model_name,

        "Accuracy": accuracy_score(
            y_val,
            predictions
        ),

        "Precision": precision_score(
            y_val,
            predictions
        ),

        "Recall": recall_score(
            y_val,
            predictions
        ),

        "F1": f1_score(
            y_val,
            predictions
        ),

        "ROC_AUC": roc_auc_score(
            y_val,
            probabilities
        )
    }

    return pipeline, results

In [7]:
trained_models = {}
model_results = []

for name, classifier in models.items():

    print(f"Training {name} ...")

    pipeline, results = train_and_evaluate(
        name,
        classifier
    )

    trained_models[name] = pipeline
    model_results.append(results)

print("\nAll models trained.")

Training Logistic Regression ...
Training Decision Tree ...
Training Random Forest ...

All models trained.


In [8]:
results_df = pd.DataFrame(
    model_results
)

results_df = (
    results_df
    .sort_values(
        "ROC_AUC",
        ascending=False
    )
    .reset_index(drop=True)
)

results_df.round(3)

,Model,Accuracy,Precision,Recall,F1,ROC_AUC
0,Random Forest,0.725,0.344,0.346,0.345,0.647
1,Logistic Regression,0.558,0.274,0.673,0.389,0.636
2,Decision Tree,0.664,0.264,0.339,0.297,0.545


In [9]:
RESULTS_PATH = (
    TABLES_DIR /
    "part_b_model_validation_results.csv"
)

results_df.to_csv(
    RESULTS_PATH,
    index=False
)

print("Saved:", RESULTS_PATH)

Saved: /Users/janakdobariya/Bramha/NLP_Engineering/BA/ai_patent_business_analytics/Outputs/tables/part_b_model_validation_results.csv
